# 03 描述统计 + 分布 + 相关性（任务书：统计特征与相关性分析）

In [ ]:
import sys
from pathlib import Path
# 让 notebook 能 import src
sys.path.append(str(Path('..').resolve()))

import pandas as pd
import numpy as np

from src.utils import load_config, ensure_dir, set_seed

cfg = load_config('../config/config.yaml')
OUT = ensure_dir('../outputs')
FIG = ensure_dir(OUT/'figures')
df = pd.read_parquet(OUT/'clean_data.parquet')
from src.utils import safe_col
from src.plots import hist_plot

bcf_col = safe_col(df, cfg['data']['columns']['target_bcf'])

if 'crop' in df.columns:
    stats = df.groupby(['crop','ph_bin'])[bcf_col].describe(percentiles=[0.05,0.25,0.5,0.75,0.95]).reset_index()
else:
    stats = df.groupby(['ph_bin'])[bcf_col].describe(percentiles=[0.05,0.25,0.5,0.75,0.95]).reset_index()

stats.to_excel(OUT/'eda_tables.xlsx', index=False)
hist_plot(df[bcf_col].to_numpy(), 'BCF distribution (all)', str(FIG/'bcf_hist_all.png'))
stats.head(20)
import matplotlib.pyplot as plt
num = df.select_dtypes(include=[np.number]).copy()
corr = num.corr(method='pearson')

plt.figure(figsize=(10,8))
plt.imshow(corr.to_numpy())
plt.title('Pearson correlation (numeric features)')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=6)
plt.yticks(range(len(corr.columns)), corr.columns, fontsize=6)
plt.tight_layout()
plt.savefig(FIG/'corr_heatmap_numeric.png', dpi=200)
plt.close()

# 初版特征清单
bcf_col = safe_col(df, cfg['data']['columns']['target_bcf'])
drop = set(cfg['data'].get('drop_from_model', [])) | {bcf_col}
features = [c for c in df.columns if c not in drop]
import json
(Path(OUT)/'feature_list.json').write_text(json.dumps({'features':features}, ensure_ascii=False, indent=2), encoding='utf-8')
len(features)
